# LoRA from scratch (NumPy)

Interactive companion to `lora.py` / `run_smoke.py`.

Pretrain a tiny MLP on task A, then adapt it to a shifted task B with frozen weights and low-rank adapters `A`, `B`. Compare against a full fine-tune and merge the adapter back into the weights.

In [ ]:
import sys, copy
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
import numpy as np
from tasks import make_teachers, make_task
from lora import init_mlp, init_lora, train, accuracy, forward, merge_lora, n_params

T = make_teachers(42, shift_rank=2, shift_scale=0.7)
XA, yA = make_task(T, 'A', 8000, 1); XAt, yAt = make_task(T, 'A', 2000, 2)
XB, yB = make_task(T, 'B', 200, 3);  XBt, yBt = make_task(T, 'B', 2000, 4)

## 1. Pretrain on task A

In [ ]:
rng = np.random.default_rng(42)
base = init_mlp(16, 64, 4, rng)
train(base, XA, yA, epochs=80, lr=3e-3, batch=64, rng=rng)
print('task A acc', accuracy(base, XAt, yAt), '| zero-shot task B acc', accuracy(base, XBt, yBt), '| params', n_params(base))

## 2. LoRA adapter (rank 4): only A and B train

In [ ]:
ad = init_lora(base, 4, np.random.default_rng(104))
print({k: v.shape for k, v in ad.items()}, 'trainable =', n_params(ad))
train(base, XB, yB, epochs=60, lr=1e-2, batch=32, rng=np.random.default_rng(7), lora=ad, scale=1.0, train_base=False)
print('task B acc with adapter', accuracy(base, XBt, yBt, ad))
print('task A acc, adapter off', accuracy(base, XAt, yAt))

## 3. Merge the adapter into the weights

In [ ]:
merged = merge_lora(base, ad, 1.0)
diff = np.abs(forward(base, XBt, ad)[0] - forward(merged, XBt)[0]).max()
print('max |logit diff|', diff, '| merged acc', accuracy(merged, XBt, yBt))
print('rank of delta W1 =', np.linalg.matrix_rank(ad['A1'] @ ad['B1']))

## 4. Full fine-tune for comparison

In [ ]:
full = copy.deepcopy(base)
train(full, XB, yB, epochs=60, lr=3e-3, batch=32, rng=np.random.default_rng(7))
print('full FT task B acc', accuracy(full, XBt, yBt), '| task A acc after', accuracy(full, XAt, yAt))